# GTEx global alignment

The SHAP results driving this analysis come from the `lv_importance_rf_true_labels_gtex` rule (`scripts/gtex/lv_importance_true_labels.py`). For each GTEx tissue, this notebook tests whether the selected SHAP latent variables recover the true tissue via over-representation analysis (ORA) against the GTEx tissue gene-set database, both for the single highest-ranked LV per tissue (top1) and for the smallest set of top-ranked LVs needed to reach `CUMULATIVE_PCT`% of cumulative SHAP per tissue (cumulative20), the same criterion used in `01_LV_importance.ipynb` and `02_b_matrix.ipynb`. Per-LV ORA follows the retry-with-fallback approach from `00_pseudobulk/02_disentangle.ipynb`: it tests the top `TOP_GENE_PCT` (1%) gene loadings first, and if nothing clears FDR < `FDR_THRESH` (5%), retries once at `TOP_GENE_PCT_FALLBACK` (3%) before giving up on that LV.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
library(here)
library(dplyr)
library(tidyr)
library(stringr)
library(Matrix)
library(clusterProfiler)
library(ggplot2)

## Settings

In [ ]:
SHAP_DIR <- here('output', '03_model_biology', '01_gtex',
                 '01_LV_importance_rf_true_labels',
                 'gtex_feature_importance_true_labels_binary_shap')
OUT_DIR  <- here('output', '03_model_biology', '01_gtex', '07_global_alignment_rf_true_labels')
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

CLAMP_RDS      <- here('output', '01_model_building', '01_gtex', 'CLAMPfull.rds')
GTEX_TISSUE_DB <- here('data', 'archs4', 'GTEx_Tissues_pathMat.rds')

N_LVS_PER_TISSUE <- 1L
CUMULATIVE_PCT   <- 20  # same threshold as 02_b_matrix.ipynb
TOP_GENE_PCT     <- 0.01
TOP_GENE_PCT_FALLBACK <- 0.03  # retried once for LVs with zero significant hits at TOP_GENE_PCT
FDR_THRESH       <- 0.05

In [ ]:
shap_all <- read.delim(file.path(SHAP_DIR, 'all_shap_positive.tsv'),
                       stringsAsFactors = FALSE, check.names = FALSE)

selected_lvs_top1 <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::slice_head(n = N_LVS_PER_TISSUE) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selected_lvs_cum <- shap_all %>%
    dplyr::arrange(Tissue, Rank) %>%
    dplyr::group_by(Tissue) %>%
    dplyr::mutate(
        reaches_thresh = Cumulative_Percent >= CUMULATIVE_PCT,
        cutoff_rank = if (any(reaches_thresh)) min(Rank[reaches_thresh]) else max(Rank)
    ) %>%
    dplyr::filter(Rank <= cutoff_rank) %>%
    dplyr::ungroup() %>%
    dplyr::rename(LV = Feature) %>%
    dplyr::select(Tissue, LV, Rank, Mean_SHAP_Tissue, Cumulative_SHAP, Cumulative_Percent)

selections <- list(top1 = selected_lvs_top1, cumulative20 = selected_lvs_cum)

cat('Tissues:', dplyr::n_distinct(selected_lvs_top1$Tissue), '\n')
for (nm in names(selections)) {
    cat(sprintf('  [%s] selected LV/tissue rows: %d, unique LVs: %d\n',
                nm, nrow(selections[[nm]]), dplyr::n_distinct(selections[[nm]]$LV)))
}

dplyr::bind_rows(lapply(names(selections), function(nm) {
    selections[[nm]] %>%
        dplyr::count(Tissue, name = 'n_selected_lvs') %>%
        dplyr::mutate(analysis = nm)
})) %>%
    tidyr::pivot_wider(names_from = analysis, values_from = n_selected_lvs) %>%
    dplyr::arrange(Tissue)


In [ ]:
clamp <- readRDS(CLAMP_RDS)
Z_full <- as.matrix(clamp$Z)
rm(clamp)

all_selected_lvs <- unique(unlist(lapply(selections, function(df) df$LV)))
selected_unique_lvs <- intersect(all_selected_lvs, colnames(Z_full))
missing_lvs <- setdiff(all_selected_lvs, colnames(Z_full))
if (length(missing_lvs) > 0) warning('Missing LVs in Z: ', paste(missing_lvs, collapse = ', '))

Z <- Z_full[, selected_unique_lvs, drop = FALSE]
rm(Z_full)
universe_genes <- rownames(Z)
n_top_genes          <- max(1L, ceiling(TOP_GENE_PCT * nrow(Z)))
n_top_genes_fallback  <- max(1L, ceiling(TOP_GENE_PCT_FALLBACK * nrow(Z)))

cat('Z:', nrow(Z), 'genes x', ncol(Z), 'unique LVs (union across analyses)\n')
cat('Top genes per LV:', n_top_genes, '(primary), ', n_top_genes_fallback, '(fallback)\n')


In [ ]:
gtex_mat <- readRDS(GTEX_TISSUE_DB)
term_names <- unname(colnames(gtex_mat))

parse_gtex_tissue <- function(x) {
    x <- sub('^GTEx_Tissues_', '', x)
    x <- sub(' (Male|Female) [0-9]+-[0-9]+ Up$', '', x)
    x
}

normalize_text <- function(x) {
    x <- tolower(x)
    x <- gsub('[^a-z0-9]+', ' ', x)
    stringr::str_squish(x)
}

term2gene <- lapply(seq_along(term_names), function(i) {
    rownames(gtex_mat)[gtex_mat[, i] != 0]
})
names(term2gene) <- term_names

term2gene_df <- do.call(rbind, lapply(names(term2gene), function(term) {
    data.frame(term = term, gene = term2gene[[term]], stringsAsFactors = FALSE)
}))

term_map <- data.frame(
    term = term_names,
    parsed_tissue = parse_gtex_tissue(term_names),
    normalized_tissue = normalize_text(parse_gtex_tissue(term_names)),
    stringsAsFactors = FALSE
)

tissue_check <- dplyr::bind_rows(selections) %>%
    dplyr::distinct(Tissue) %>%
    dplyr::mutate(
        normalized_tissue = normalize_text(Tissue),
        matched_in_gtex_db = normalized_tissue %in% term_map$normalized_tissue,
        n_db_sets = vapply(normalized_tissue, function(x) sum(term_map$normalized_tissue == x, na.rm = TRUE), integer(1))
    )

cat('GTEx tissue gene sets:', length(term2gene), '\n')
stopifnot(all(tissue_check$matched_in_gtex_db))
tissue_check

In [ ]:
# ORA per selected LV against the GTEx tissue gene-set database -- top 1%
# gene loadings (TOP_GENE_PCT) tested via clusterProfiler::enricher(). If
# NOTHING clears FDR < FDR_THRESH at 1%, retries once at
# TOP_GENE_PCT_FALLBACK (3%) before giving up on that LV -- same
# retry-with-fallback used for module-level marker recovery in
# 00_pseudobulk/02_disentangle.ipynb.
run_gtex_tissue_ora <- function(genes, universe) {
    res <- tryCatch(
        clusterProfiler::enricher(
            gene          = genes,
            universe      = universe,
            TERM2GENE     = term2gene_df,
            pAdjustMethod = 'BH',
            pvalueCutoff  = 1,
            qvalueCutoff  = 1,
            minGSSize     = 10,
            maxGSSize     = 500
        ),
        error = function(e) NULL
    )
    if (is.null(res)) return(NULL)
    df <- as.data.frame(res)
    if (nrow(df) == 0) return(NULL)
    df
}

get_top_genes <- function(lv, pct) {
    vals    <- Z[, lv]
    n_genes <- max(1L, ceiling(pct * length(vals)))
    universe_genes[order(vals, decreasing = TRUE)[seq_len(n_genes)]]
}

run_lv_ora_with_fallback <- function(lv) {
    df       <- run_gtex_tissue_ora(get_top_genes(lv, TOP_GENE_PCT), universe_genes)
    sig      <- if (!is.null(df)) df[df$p.adjust < FDR_THRESH, ] else df
    pct_used <- TOP_GENE_PCT

    if (is.null(sig) || nrow(sig) == 0) {
        df       <- run_gtex_tissue_ora(get_top_genes(lv, TOP_GENE_PCT_FALLBACK), universe_genes)
        pct_used <- TOP_GENE_PCT_FALLBACK
    }
    if (is.null(df)) return(NULL)
    df$top_gene_pct_used <- pct_used
    df
}

ora_list <- lapply(colnames(Z), function(lv) {
    df <- run_lv_ora_with_fallback(lv)
    if (is.null(df)) return(NULL)
    df$LV <- lv
    df
})

ora_all <- do.call(rbind, Filter(Negate(is.null), ora_list))
if (is.null(ora_all)) {
    ora_all <- data.frame()
} else {
    rownames(ora_all) <- NULL
    ora_all <- ora_all %>%
        dplyr::left_join(term_map, by = c('ID' = 'term')) %>%
        dplyr::select(LV, ID, Description, parsed_tissue, normalized_tissue,
                      GeneRatio, BgRatio, pvalue, p.adjust, qvalue, geneID, Count,
                      top_gene_pct_used)
}

write.csv(ora_all, file.path(OUT_DIR, 'gtex_tissue_ora_per_lv.csv'), row.names = FALSE)
cat('ORA rows:', nrow(ora_all), '\n')
cat('LVs retried at fallback pct:',
    dplyr::n_distinct(ora_all$LV[ora_all$top_gene_pct_used == TOP_GENE_PCT_FALLBACK]),
    '/', dplyr::n_distinct(ora_all$LV), '\n')
head(ora_all)


In [ ]:
run_alignment_summary <- function(selected_lvs, ora_all, out_dir) {
    dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
    sig_ora <- ora_all %>% dplyr::filter(LV %in% selected_lvs$LV, p.adjust < FDR_THRESH)

    detail <- selected_lvs %>%
        dplyr::rowwise() %>%
        dplyr::mutate(
            normalized_true_tissue = normalize_text(Tissue),
            n_sig_terms = sum(sig_ora$LV == LV),
            n_true_terms = sum(sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue),
            tissue_correct = n_true_terms > 0,
            best_any_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            best_true_padj = {
                x <- sig_ora$p.adjust[sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue]
                if (length(x) == 0) NA_real_ else min(x, na.rm = TRUE)
            },
            matched_terms = paste(sig_ora$ID[sig_ora$LV == LV & sig_ora$normalized_tissue == normalized_true_tissue], collapse = ' | ')
        ) %>%
        dplyr::ungroup()

    tissue_summary <- detail %>%
        dplyr::group_by(Tissue) %>%
        dplyr::summarise(
            n_selected_lvs = dplyr::n(),
            tissue_correct = any(tissue_correct),
            correct_lvs = paste(LV[tissue_correct], collapse = ';'),
            best_true_padj = if (all(is.na(best_true_padj))) NA_real_ else min(best_true_padj, na.rm = TRUE),
            .groups = 'drop'
        ) %>%
        dplyr::mutate(correct_score = as.integer(tissue_correct))

    final_pct_tissue_correct <- 100 * mean(tissue_summary$tissue_correct)
    final_summary <- data.frame(
        n_tissues = nrow(tissue_summary),
        n_tissues_correct = sum(tissue_summary$tissue_correct),
        pct_tissue_correct = final_pct_tissue_correct,
        stringsAsFactors = FALSE
    )

    write.csv(detail, file.path(out_dir, 'gtex_global_alignment_detail.csv'), row.names = FALSE)
    write.csv(tissue_summary, file.path(out_dir, 'gtex_global_alignment_summary.csv'), row.names = FALSE)
    write.csv(final_summary, file.path(out_dir, 'gtex_global_alignment_final_pct.csv'), row.names = FALSE)

    list(detail = detail, tissue_summary = tissue_summary, final_summary = final_summary)
}

results <- lapply(names(selections), function(nm) {
    run_alignment_summary(selections[[nm]], ora_all, file.path(OUT_DIR, nm))
})
names(results) <- names(selections)

dplyr::bind_rows(lapply(names(results), function(nm) {
    results[[nm]]$final_summary %>% dplyr::mutate(analysis = nm, .before = 1)
}))


In [ ]:
cat('--- top1 ---\n')
results$top1$tissue_summary %>% dplyr::arrange(Tissue)

cat('--- cumulative20 ---\n')
results$cumulative20$tissue_summary %>% dplyr::arrange(Tissue)


In [ ]:
detail_cols <- c('Tissue', 'LV', 'Rank', 'Mean_SHAP_Tissue', 'Cumulative_Percent',
                 'tissue_correct', 'n_sig_terms', 'n_true_terms',
                 'best_true_padj', 'matched_terms')

cat('--- top1 ---\n')
results$top1$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)

cat('--- cumulative20 ---\n')
results$cumulative20$detail %>% dplyr::select(dplyr::all_of(detail_cols)) %>% dplyr::arrange(Tissue, Rank)


In [ ]:
# Per-LV correctness within the cumulative20 selection (not just "any LV correct")
per_lv_summary <- results$cumulative20$detail %>%
    dplyr::group_by(Tissue) %>%
    dplyr::summarise(
        n_lvs = dplyr::n(),
        n_correct = sum(tissue_correct),
        pct_correct = 100 * n_correct / n_lvs,
        .groups = "drop"
    ) %>%
    dplyr::arrange(pct_correct)

print(per_lv_summary, n = Inf)

total_lvs <- sum(per_lv_summary$n_lvs)
total_correct <- sum(per_lv_summary$n_correct)
cat(sprintf(
    "\nOverall: %d/%d LV-tissue rows correct (%.1f%%) across cumulative20 selection\n",
    total_correct, total_lvs, 100 * total_correct / total_lvs
))
cat(sprintf("Tissues at 100%% LV concordance: %d/%d\n",
            sum(per_lv_summary$pct_correct == 100), nrow(per_lv_summary)))

write.csv(per_lv_summary, file.path(OUT_DIR, "cumulative20", "gtex_global_alignment_per_lv_pct.csv"), row.names = FALSE)


In [ ]:
# Tissue-group-level gene (Z matrix) recovery -- the companion to
# 02_b_matrix.ipynb's grouped B-matrix heatmap (Panel G in supp1.ipynb).
# Subtissues are merged into their parent anatomical tissue with the same
# convention used there (Cells - * kept separate; everything else grouped
# on the text before " - "), and a group counts as recovered if ANY member
# subtissue's cumulative20-selected LVs were significantly ORA-enriched
# for the true tissue.
get_heatmap_group <- function(name) {
    if (startsWith(name, "Cells - ")) return(name)
    parts <- strsplit(name, " - ")[[1]]
    if (length(parts) > 1) trimws(parts[1]) else name
}

grouped_summary <- results$cumulative20$tissue_summary %>%
    dplyr::mutate(Tissue_Group = vapply(Tissue, get_heatmap_group, character(1))) %>%
    dplyr::group_by(Tissue_Group) %>%
    dplyr::summarise(
        n_subtissues = dplyr::n(),
        tissue_correct = any(tissue_correct),
        best_true_padj = if (all(is.na(best_true_padj))) NA_real_ else min(best_true_padj, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    dplyr::arrange(Tissue_Group)

n_grp_correct <- sum(grouped_summary$tissue_correct)
n_grp_total   <- nrow(grouped_summary)
cat(sprintf("Gene-level (Z matrix) tissue-group recovery: %d/%d (%.1f%%)\n",
            n_grp_correct, n_grp_total, 100 * n_grp_correct / n_grp_total))

write.csv(grouped_summary, file.path(OUT_DIR, "gtex_global_alignment_grouped_summary.csv"), row.names = FALSE)
grouped_summary


## Summary of results

In [10]:
stopifnot(file.exists(file.path(OUT_DIR, 'gtex_tissue_ora_per_lv.csv')))

for (nm in names(selections)) {
    out_dir <- file.path(OUT_DIR, nm)
    expected_rows <- nrow(selections[[nm]])
    stopifnot(nrow(results[[nm]]$detail) == expected_rows)
    stopifnot(nrow(results[[nm]]$tissue_summary) == dplyr::n_distinct(shap_all$Tissue))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_detail.csv')))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_summary.csv')))
    stopifnot(file.exists(file.path(out_dir, 'gtex_global_alignment_final_pct.csv')))
    cat(sprintf(
        '[%s] Checks passed. Denominator = %d tissues and %d selected LV/tissue rows. Final %% tissue correct = %.2f\n',
        nm, nrow(results[[nm]]$tissue_summary), nrow(results[[nm]]$detail),
        results[[nm]]$final_summary$pct_tissue_correct
    ))
}


[top1] Checks passed. Denominator = 49 tissues and 49 selected LV/tissue rows. Final % tissue correct = 87.76
[cumulative25] Checks passed. Denominator = 49 tissues and 226 selected LV/tissue rows. Final % tissue correct = 100.00
